##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 Torch XLA 和 Hugging Face TRL 微調 Gemma

歡迎閱讀本關於使用 [Torch XLA](https://github.com/pytorch/xla) 的 fine-tuning 和 [Gemma](https://huggingface.co/google/gemma-2b) 的逐步指南。

[**Gemma**](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開放模型，採用與創建 Gemini 模型相同的研究和技術構建。它們是文字到文字、僅限解碼器的大型語言模型，提供英文版本，具有開放權重、預訓練變體和指令調整變體。 Gemma 模型非常適合各種文本生成任務，包括問答、摘要和推論。它們的尺寸相對較小，因此可以將它們部署在資源有限的環境中，例如筆記型電腦、桌上型電腦或您自己的雲端基礎設施，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
[**Torch XLA**](https://pytorch.org/xla/) 讓您能夠利用 TPU（張量處理單元）的運算能力來高效訓練深度學習模型。透過將 PyTorch 與 [XLA（加速線性代數）](https://openxla.org/xla) 編譯器連接，Torch XLA 將 PyTorch 操作轉換為可以在 TPU 上執行的 XLA 操作。這意味著您可以像往常一樣在 PyTorch 中編寫模型，而 Torch XLA 會處理底層計算以在 TPU 上高效執行它們。
[**Transformer 強化學習 (TRL)**](https://github.com/huggingface/trl) 是 Hugging Face 開發的 framework，使用監督微調 (SFT)、獎勵建模 (RM)、近端策略最佳化 (PPO)、直接偏好最佳化 (DPO) 等方法來微調和擴散程式對齊語言和虛擬語言和擴散器 Transformer。
將 PyTorch 與 XLA 集成，使開發人員可以在 TPU 上執行 PyTorch 程式碼，而只需對其現有程式碼庫進行最小程度的更改。這種無縫整合提供了 TPU 的效能優勢，同時保持了 PyTorch framework 的靈活性和易用性。
在本notebook結束時，您將了解到：
- 關於火炬 XLA
- 如何使用 Hugging Face 的 **TRL** framework、**Torch XLA** 和 TPU 在 [Gemma 2 2B](https://huggingface.co/google/gemma-2-2b) 上執行 **參數高效微調 (PEFT)** 和 **低階適應 (@@P0004@)。

<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Finetune_with_Torch_XLA.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>
<br><br>
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/notebooks/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_2]Finetune_with_Torch_XLA.ipynb)

## 設定

### 選擇執行時環境

首先，您可以選擇 **Google Colab** 或 **Kaggle** 作為您的平台。選擇一個，然後從那裡繼續。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>

  1. 按一下「**在 Colab** 中開啟」。
  2. In the menu, go to **Runtime** > **Change runtime type**.
  3. 在 **硬體加速器** 下，選擇 **TPU**。
  4. 確保 **TPU 類型** 設定為 **TPU v2-8**。

- #### **Kaggle** <img src="https://upload.wikimedia.org/wikipedia/commons/7/7c/Kaggle_logo.png" alt="Kaggle" width="40"/>

  1. 按一下「**在 Kaggle** 中開啟」。
  2. 點選右側邊欄中的**設定**。
  3. 在 **加速器** 下，選擇 **TPU**。
- 注意：Kaggle 目前提供 **TPU v3-8**。  4. 儲存設置，notebook 將在 TPU 支援下重新啟動。


### Gemma 使用Hugging Face

在深入學習本教學之前，讓我們先設定Gemma：
1. **建立一個 Hugging Face 帳戶**：如果您沒有帳戶，您可以[此處]註冊一個免費帳戶(https://huggingface.com/join)。
2. **造訪Gemma型號**：造訪[Gemma型號頁面](https://huggingface.com/collections/google/gemma-2-release-667d6600fd5220e7b967f315)並接受使用條件。
3. **產生Hugging Facetoken**：前往您的Hugging Face [設定頁面](https://huggingface.com/settings/tokens)並產生新的存取token（最好具有`write`權限）。在本教學的後面部分，您將需要這個token。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的憑證

要存取私有模型和dataset，您需要登入Hugging Face（HF）生態系統。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>
如果您使用 Colab，您可以使用 Colab Secrets manager 安全地儲存 Hugging Face token (`HF_TOKEN`)：  1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
  2. **新增Hugging Facetoken**：
- 建立一個新的secret，其**名稱**為`HF_TOKEN`。 - 將token 金鑰複製/貼上到`HF_TOKEN` 的**值** 輸入框中。 - **切換**左側的按鈕以允許notebook訪問secret
- #### **Kaggle** <img src="https://upload.wikimedia.org/wikipedia/commons/7/7c/Kaggle_logo.png" alt="Kaggle" width="40"/>
要在此 notebook 中安全地使用 Hugging Face token (`HF_TOKEN`)，您需要將其作為 secret 添加到 Kaggle 環境中：  1. 開啟 Kaggle notebook 並找到 notebook 介面頂部的 **插件** 選單。
  2. 點選 **Secrets** 來管理您的環境secrets。
<img src="https://i.imgur.com/vxrtJuM.png" alt="The Secrets option is found at the top." width=50%>  3. **新增Hugging Facetoken**：
- 點選「**新增secret**」按鈕。 - 在**標籤**欄位中，輸入`HF_TOKEN`。 - 在 **值** 欄位中，貼上 Hugging Face token。 - 點選「**儲存**」新增secret。

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    # Running on Colab
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
elif os.path.exists('/kaggle/working'):
    # Running on Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")
else:
    # Not running on Colab or Kaggle
    raise EnvironmentError('This notebook is designed to run on Google Colab or Kaggle.')

This code retrieves your secrets and sets them as environment variables, which you will use later in the tutorial.

### 設定環境

接下來，您將使用 Torch XLA 在 TPU VM 上安裝 fine-tuning 和 Gemma 模型所需的所有 Python 軟體包來設定環境。

In [ ]:
# Uninstalling any existing TensorFlow installations and then install the CPU-only version to avoid conflicts while using the TPU.
!pip uninstall -y tensorflow tf-keras
!pip install tensorflow==2.18.0 tf-keras==2.18.0

!pip uninstall tensorflow -y
!pip install tensorflow-cpu==2.18.0 -q

# Install the appropriate Hugging Face libraries to ensure compatibility with the Gemma model and PEFT.
!pip install transformers==4.46.1 -U -q
!pip install datasets==3.1.0 -U -q
!pip install trl==0.12.0 peft==0.13.2 -U -q
!pip install accelerate==0.34.0 -U -q

# Install PyTorch and Torch XLA with versions compatible with the TPU runtime, ensuring efficient TPU utilization.
!pip install -qq torch~=2.5.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -qq torch_xla[tpu]~=2.5.0 -f https://storage.googleapis.com/libtpu-releases/index.html

# Install the `tpu-info` package to display TPU-related information
!pip install tpu-info

Found existing installation: tensorflow 2.15.0
Uninstalling tensorflow-2.15.0:
  Successfully uninstalled tensorflow-2.15.0
Found existing installation: tf_keras 2.15.1
Uninstalling tf_keras-2.15.1:
  Successfully uninstalled tf_keras-2.15.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.3/615.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 311.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.3/381.3 kB 24.4 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.2.0
    Uninstalling ml-dtypes-0.2.0:
      Successfully uninstalled ml-dtypes-0.2.0
  Attempting uninstall: tensorboard
    Fou

**注意**：確保您的 PyTorch 和 Torch XLA 版本與您正在使用的 TPU 相容。

### 驗證 TPU 設定


您執行 `!tpu-info` 來驗證 TPU 是否已正確初始化。

In [ ]:
!tpu-info

TPU Chips                                     
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━┓
┃ Chip        ┃ Type        ┃ Devices ┃ PID  ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━┩
│ /dev/accel0 │ TPU v2 chip │ 2       │ None │
│ /dev/accel1 │ TPU v2 chip │ 2       │ None │
│ /dev/accel2 │ TPU v2 chip │ 2       │ None │
│ /dev/accel3 │ TPU v2 chip │ 2       │ None │
└─────────────┴─────────────┴─────────┴──────┘
Libtpu metrics unavailable. Is there a framework using the TPU? See https://github.com/google/cloud-accelerator-diagnostics/tree/main/tpu_info for more information


如果一切設定正確，您應該會看到列印的 TPU 詳細資訊。

## 導入庫

現在，導入 fine-tuning 所需的所有必要庫。

In [ ]:
import pandas as pd

import torch
print(f"PyTorch version: {torch.__version__}")

import torch_xla
print(f"Torch XLA version: {torch_xla.__version__}")

import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, PeftModel

from datasets import load_dataset

# Enable Single Program Multiple Data (SPMD) mode,
# which allows for parallel execution across multiple TPU cores
xr.use_spmd()

PyTorch version: 2.5.1+cpu
Torch XLA version: 2.5.1+libtpu


此設定可確保您的環境正確設定為使用帶有 PyTorch 的 TPU。

## 使用 PEFT 和 LoRA 進行微調

Gemma 等大型語言模型 (LLM) 的傳統 fine-tuning 需要調整數十億個參數，從而佔用大量資源。此過程需要大量的運算能力和時間，這對於許多用例來說可能不切實際。這就是參數高效微調 (PEFT) 技術的用武之地。
### 參數高效率微調 (PEFT)

PEFT 可讓您透過僅更新一小部分參數來使大型模型適應特定任務。 PEFT 不是重新訓練整個模型，而是增加輕量級層或轉接器。大多數預先訓練的權重保持凍結狀態。這種方法大大降低了 fine-tuning 所需的計算要求和資料量，使得即使在中等硬體上也可以微調大型模型。
### 低階適應 (LoRA)

在這些技術中，一個有效的選擇是低階適應（LoRA）。 LoRA 將小型可訓練矩陣引入模型架構中，專門針對 Transformer 模型的注意力層。 LoRA 沒有更新完整的權重矩陣，而是添加了秩分解矩陣，使適應更有效率。
#### LoRA的主要優勢

- **效率**：LoRA 透過使用低秩自適應顯著減少了可訓練參數的數量，使fine-tuning 過程更加高效。
- **記憶體節省**：由於僅更新額外的低秩矩陣，因此 GPU/TPU 記憶體需求顯著降低。
- **模組化**：LoRA 適配器可以輕鬆地與原始模型合併或保持獨立，從而提供部署靈活性。

在下一節中，您將探索如何使用 LoRA 實作 PEFT，以在 TPU 上使用 Torch XLA 微調 Gemma 並執行以下步驟：
- 加載dataset
- 設定訓練參數
- 載入Gemma模型和tokenizer
- 使用 **TRL** 的 `SFTTrainer` 類別在 TPU 上微調模型

### 加載dataset

對於本指南，您將使用Hugging Face 中的現有dataset。如果您願意，可以將其替換為您自己的dataset。
本指南選擇的 dataset 是 [**hieunguyenminh/roleplay**](https://huggingface.com/datasets/hieunguyenminh/roleplay)，它體現了各種原創角色，每個角色都有獨特的性格。它包括虛構的人物，有他們自己的背景、核心特徵、關係、目標和獨特的說話風格。
**學分：** **https://huggingface.com/hieunguyenminh**

您指定dataset 名稱並使用Hugging Face `datasets` library 中的`load_dataset` 函數來載入dataset 的訓練分割。

In [ ]:
dataset_name = "hieunguyenminh/roleplay"
dataset = load_dataset(dataset_name, split="train")
dataset

README.md:   0%|          | 0.00/3.58k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.15M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5755 [00:00<?, ? examples/s]

Dataset({
    features: ['name', 'description', 'text'],
    num_rows: 5755
})

讓我們看幾個範例來了解數據。

In [ ]:
dataset[10]['text']

'<|system|>Michael Jordan, also known as "MJ" or "His Airness," is a basketball legend renowned for his unparalleled competitive spirit and extraordinary athletic prowess. Born on February 17, 1963, in Brooklyn, New York, he grew up in Wilmington, North Carolina. Jordan\'s illustrious career in the NBA, primarily with the Chicago Bulls, saw him secure six championship wins and earn numerous accolades, including five regular-season MVP awards. His impact on the global sports landscape is immeasurable, as he transcended the game of basketball to become a cultural icon. His Air Jordan sneakers, in collaboration with Nike, revolutionized the concept of athlete endorsements and remain highly coveted to this day. Jordan\'s relentless pursuit of excellence and his ability to perform under pressure have solidified his legacy as one of the greatest athletes of all time.</s>\n<|user|>What was Michael Jordan\'s mindset during high-pressure moments in games?</s>\n<|assistant|>Michael Jordan\'s min

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import data_table

    # Enable interactive DataFrame display
    data_table.enable_dataframe_formatter()

# Convert the 'train' split to a Pandas DataFrame
df = pd.DataFrame(dataset)

# Select the 'text' column and exclude the rest
df_text = df[['text']]
df_text.head(5)

首先，讓我們將 dataset 分成訓練集和驗證集。

In [ ]:
# The first 80% of `train` for training
train_dataset = load_dataset(dataset_name, split='train[:80%]')

# The last 20% of `train` for evaluation
valid_dataset = load_dataset(dataset_name, split='train[-20%:]')

In [ ]:
train_dataset

Dataset({
    features: ['name', 'description', 'text'],
    num_rows: 4604
})

In [ ]:
valid_dataset

Dataset({
    features: ['name', 'description', 'text'],
    num_rows: 1151
})

對dataset進行預處理以進行[Gemma指令調整](https://ai.google.dev/gemma/docs/formatting)。
**注意**：Gemma 不支援對話中的`system` 角色。相反，您將用 `user` 角色替換它。

In [ ]:
def convert_to_gemma_format(text):
  # Replace role tokens with Gemma's instruction tuning format
    text = text.replace("<|system|>", "<start_of_turn>user\n")
    text = text.replace("<|assistant|>", "<start_of_turn>model\n")
    text = text.replace("<|user|>", "<start_of_turn>user\n")

    # Replace end-of-sequence tokens with <end_of_turn>
    text = text.replace("</s>", "<end_of_turn>\n")

    # Clean up extra newlines if necessary
    text = text.strip()
    return text

def preprocess_function(example):
    text = example["text"]
    text = convert_to_gemma_format(text)

    return {"text": text}

In [ ]:
# Apply the preprocessing
train_dataset = train_dataset.map(preprocess_function,
                                  remove_columns=list(train_dataset.features))
train_dataset

In [ ]:
valid_dataset = valid_dataset.map(preprocess_function,
                                  remove_columns=list(valid_dataset.features))
valid_dataset

### 訓練設定

現在您需要定義 fine-tuning 流程所需的所有超參數和設定，其中包括定義以下內容：
- The base model and new model names
- LoRA Configuration
- 訓練參數
- SFT Parameters
- 雜項。參數


首先指定基本模型 (`google/gemma-2b`) 和保存微調模型的目錄 (`gemma-ft`)。

In [ ]:
# Define model names
model_name = "google/gemma-2-2b-it"
new_model = "gemma-ft"

LoRA（低階適應）透過僅適應模型參數的子集來實現高效的fine-tuning。
在這裡，您設定以下參數：- `lora_r`到64，它控制適應矩陣的等級，
- `lora_alpha` 至 32 進行縮放
- `lora_dropout` 為 0.1 以防止過度擬合。

In [ ]:
# LoRA attention dimension
lora_r = 64 # @param {"type":"slider","min":0,"max":64,"step":2}
# Alpha parameter for LoRA scaling
lora_alpha = 32 # @param {"type":"slider","min":0,"max":64,"step":2}
# Dropout probability for LoRA layers
lora_dropout = 0.1 # @param {"type":"slider","min":0,"max":1,"step":0.01}

設定定義如何訓練模型的訓練參數。
在這裡，您將定義用於訓練和評估的**輸出目錄**、**訓練週期數**和**批量大小**。您啟用**梯度checkpointing**以節省內存，並設定`max_grad_norm`進行梯度裁剪以穩定訓練。 **學習率**、**優化器**和**學習率調度程序**設定為優化訓練過程。 `max_steps` 設定為 **-1** 以讓紀元數控制訓練持續時間。

In [ ]:
# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results" # @param {"type":"string"}
# Number of training epochs
num_train_epochs = 5 # @param {"type":"slider","min":1,"max":20,"step":2}
# Batch size per TPU core for training
per_device_train_batch_size = 32 # @param {"type":"slider","min":1,"max":64,"step":1}
# Batch size per TPU core for evaluation
per_device_eval_batch_size = 32 # @param {"type":"slider","min":1,"max":64,"step":1}
# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1 # @param {"type":"slider","min":0,"max":16,"step":2}
# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3 # @param {"type":"slider","min":0,"max":1,"step":0.01}
# Initial learning rate (adafactor optimizer)
learning_rate = 0.0001 # @param {"type":"slider","min":0.00001,"max":0.0005,"step":0.00001}
# Optimizer to use
optim = "adafactor" # adafactor, adamw_torch_fused
# Learning rate schedule (constant a bit better than cosine)
lr_scheduler_type = "constant"
# Number of training steps (overrides num_train_epochs)
max_steps = -1
# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03 # @param {"type":"slider","min":0,"max":0.1,"step":0.01}
# Enable bfloat16 precision
bf16 = True
# Log every X updates steps
logging_steps = 1

在 SFT 參數中，`max_seq_length` 設定為 512 以定義輸入的最大 token 長度，並啟用 `packing` 將多個較短序列打包到一個輸入中以提高效率。

In [ ]:
# Maximum sequence length to use
max_seq_length = 512 # @param {"type":"slider","min":32,"max":1024,"step":2}
# Pack multiple short examples in the same input sequence to increase efficiency
packing = True

### 引擎蓋下：PyTorch 和 XLA

PyTorch程式使用其`autograd`系統動態定義計算圖。 TPU 不直接執行Python程式碼；相反，它執行由 PyTorch 程式定義的計算圖。在幕後，一個名為 **XLA（加速線性代數編譯器）** 的編譯器將 PyTorch 計算圖轉換為 TPU 機器碼。該編譯器還對您的程式碼和記憶體佈局執行許多高級最佳化。當任務傳送到 TPU 時，編譯會自動發生，您無需在建置鏈中明確包含 XLA。
<img src="https://storage.googleapis.com/gweb-cloudblog-publish/images/1_PyTorchXLA_stack_diagram.max-800x800.png" alt="PyTorch and XLA 2.3 from https://cloud.google.com/blog/products/ai-machine-learning/introducing-pytorch-xla-2-3" width=50%>
**PyTorch** 和 **XLA** 的組合提供了幾個關鍵優勢：
1. **無縫的效能增強：** 保持PyTorch 直覺且 Python 的工作流程，同時透過 XLA 編譯器輕鬆實現顯著的效能提升。這種整合允許您優化模型，而無需改變您熟悉的編碼實踐。

2. **全面的生態系統存取：** 利用PyTorch 廣泛的生態系統，包括廣泛的工具、預訓練模型和充滿活力的社區。這種訪問使您能夠加速開發、利用最先進的資源並從集體專業知識中受益。

利用這些優勢，您可以使用 TPU 有效地微調您的自訂 Gemma 模型。

### 使用 TPU 微調Gemma

訓練利用 PyTorch、XLA 和 TPU 進行高效計算，並使用 LoRA 進行參數高效fine-tuning，從而透過僅調整特定層來減少可訓練參數的數量。
在這裡，您將進行以下設定：
- Gemma **基本型號** 和 **tokenizer**
- **LoRA（低階適應）** **PEFT（參數高效率微調）** 設定
- 用於高效 TPU 訓練的 [**FSDP**](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html#how-fsdp-works) 設定
- 使用訓練和 SFT 參數的 **Hugging Face `SFTTrainer` 實例**

First, load the Gemma 2B pre-trained model weights using `AutoModelForCausalLM`, while setting `torch_dtype` to `torch.bfloat16` for optimal performance on TPUs

In [ ]:
# Load the Gemma pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16
)

# You must disable the cache to prevent issues during training
model.config.use_cache = False

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

接下來，使用Hugging Face 中的`AutoTokenizer` 來載入Gemma tokenizer。您可以在此處調整tokenizer 的填充側（以及token，如果適用），以確保訓練期間的兼容性。

In [ ]:
# Load the Gemma tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# You adjust the tokenizer's padding side to ensure compatibility during TPU
# training.
tokenizer.padding_side = 'right'

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

現在，您已經載入了基本Gemma 模型和tokenizer，並設定了fine-tuning 的設定。讓我們重點關注初始化 **LoRA** 設定。由於您使用的是 LoRA，PEFT library 提供了一個方便的 [LoraConfig](https://huggingface.com/docs/peft/main/en/package_reference/lora#peft.LoraConfig)，它定義了在基礎模型的哪些層上應用適配器。通常將 LoRA 套用於 **Transformer** 的注意力層的線性投影矩陣。然後，您可以將此設定提供給稍後教學中的 `SFTTrainer` 類別。
`LoraConfig` 使用先前定義的LoRA 參數進行初始化，指定模型中的目標模組（`k_proj` 和`v_proj`）以應用LoRA 調整。

In [ ]:
# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj"
    ]
)

**完全分片資料並行 (FSDP)** 設定在 `fsdp_config` 中設置，啟用 [**全模型分片**](https://pytorch.org/docs/stable/fsdp.html#torch.distributed.fsdp.ShardingStrategy) 和 [**梯度 checkpointing**](https://huggingface.co/docs/transformers/v4.19.4/en/performance#gradient-checkpointing) 以提高 TPU 上的內存效率，並指定應啟用梯度。

In [ ]:
# Set up the FSDP config. To enable FSDP via SPMD, set xla_fsdp_v2 to True.
fsdp_config = {
    "fsdp_transformer_layer_cls_to_wrap": [
        "Gemma2DecoderLayer"
    ],
    "xla": True,
    "xla_fsdp_v2": True,
    "xla_fsdp_grad_ckpt": True
}

The `SFTConfig` is then initialized with all the training parameters defined earlier, including optimizer settings, learning rate, and logging configurations, and specifying that logs should be reported to `TensorBoard`.

In [ ]:
# Set training parameters
training_arguments = SFTConfig(
    output_dir=output_dir,
    overwrite_output_dir=True,
    save_strategy="no",
    # Training
    num_train_epochs=num_train_epochs,
    # This is the global train batch size for SPMD
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    # Required for SPMD
    dataloader_drop_last=True,
    fsdp="full_shard",
    fsdp_config=fsdp_config,
    learning_rate=learning_rate,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    },
    group_by_length=True,
    packing=packing,
    # Evaluation
    evaluation_strategy="epoch",
    # This is the global eval batch size for SPMD
    per_device_eval_batch_size=per_device_eval_batch_size,
    # Logging
    logging_steps=logging_steps,
    report_to="none",
    seed=42
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


最後，定義 TRL library 中可用的 [SFTTrainer](https://huggingface.com/docs/trl/sft_trainer)。該類繼承自Transformers library 中可用的`Trainer` 類，但專門針對受監督fine-tuning（指令調整）進行了優化。它可用於在一個或多個 GPU/TPU 上進行開箱即用的訓練，使用 [Accelerate](https://huggingface.com/docs/accelerate/index) 作為後端。
最值得注意的是，它支援[打包](https://huggingface.co/docs/trl/sft_trainer#packing-dataset--constantlengthdataset-)，將多個簡短範例打包在同一輸入序列中以提高訓練效率。

In [ ]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    peft_config=peft_config,
    args=training_arguments,
    tokenizer=tokenizer
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

現在，讓我們透過呼叫`trainer.train()`來啟動fine-tuning進程，它使用`SFTTrainer`來處理訓練循環，包括資料載入、前向和後向傳遞以及優化器步驟，所有這些都根據您提供的設定進行設定。

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py:1810: UserWarning: For backward hooks to be called, module output should be a Tensor or a tuple of Tensors but received <class 'transformers.modeling_outputs.CausalLMOutputWithPast'>
  warnings.warn("For backward hooks to be called,"
/usr/local/lib/python3.10/dist-packages/torch_xla/utils/checkpoint.py:183: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  torch.cuda.amp.autocast(**ctx.gpu_autocast_kwargs), \
/usr/local/lib/python3.10/dist-packages/torch_xla/utils/checkpoint.py:184: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):


Epoch,Training Loss,Validation Loss
1,0.621100,1.326517
2,0.464800,1.306985
3,0.324200,1.493107
4,0.253900,1.817325
5,0.250000,1.548943


/usr/local/lib/python3.10/dist-packages/torch_xla/core/xla_model.py:1457: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  xldata.append(torch.load(xbio))
Trainer.tokenizer is 

TrainOutput(global_step=700, training_loss=0.49772391183035714, metrics={'train_runtime': 2675.5038, 'train_samples_per_second': 8.372, 'train_steps_per_second': 0.262, 'total_flos': 1.842971582398464e+17, 'train_loss': 0.49772391183035714, 'epoch': 5.0})

訓練完成後，儲存微調後的模型，透過`trainer.model.to('cpu')`將其移至 CPU 以確保相容性，然後呼叫`save_pretrained(new_model)`將模型權重和設定檔儲存到`new_model`（**gemma-ft**）指定的目錄中。這允許您稍後重新加載並使用微調後的模型進行 inference 或進一步培訓。

In [ ]:
# Remove the model weights directory if it exists
!rm -rf gemma-ft

# Save the LoRA adapter
trainer.model.to('cpu').save_pretrained(new_model)

## 提示使用新微調的模型


現在您終於微調了自訂 Gemma 模型，讓我們重新載入 LoRA 適配器權重以最終 prompt 使用它，並驗證它是否真正按預期工作。

為此，請使用以下步驟正確地重新加載適配器重量：
- 使用 `AutoModelForCausalLM.from_pretrained` 首先載入 **基本 Gemma 模型**，同時設定 `low_cpu_mem_usage=True` 以優化記憶體消耗（因為您使用的是 TPU），並設定 `torch_dtype=torch.bfloat16` 以與微調模型保持一致。

- Load the **fine-tuned LoRA adapter** that you've previously saved into the base model using `PeftModel.from_pretrained`, where `new_model` is the directory containing your fine-tuned weights.

- `model.merge_and_unload` 函數**將**LoRA 適配器權重**與**基本模型權重**合併並卸載適配器，從而產生一個可用於inference 的獨立模型。

In [ ]:
# Reload the fine-tuned Gemma model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

您重新加載tokenizer以確保它與模型設定匹配，並像以前一樣調整填充側。

In [ ]:
# Reload tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = 'right'

現在，使用範例 prompt 測試微調模型，首先使用 tokenizer 產生輸入 id，然後依靠重新載入的微調模型使用 `model.generate()` 產生回應。

In [ ]:
input_text = """\
  <|system|>Introducing Minami "Echo" Ishikawa, a mysterious VR assassin known for her uncanny ability to blend seamlessly into the shadows. \
  Minami possesses a deep understanding of stealth techniques, allowing her to silently eliminate her targets with calculated precision. \
  Her cold and calculating demeanor makes her a formidable force to be reckoned with, leaving enemies shivering at the thought of facing her wrath.</s>
  <|user|>Echo, what makes you so adept at disappearing into thin air?</s>
  <|assistant|>"""

In [ ]:
input_text = convert_to_gemma_format(input_text)

In [ ]:
input_ids = tokenizer(input_text, return_tensors="pt").to("cpu")
outputs = model.generate(**input_ids, max_length=512,
                         eos_token_id=tokenizer.eos_token_id)

最後，您使用 `tokenizer.decode` 將輸出 tokens 解碼回人類可讀的文字並列印結果，以便您可以查看微調後的模型如何回應 prompt。

In [ ]:
print(tokenizer.decode(outputs[0]))

<bos><start_of_turn>user
Introducing Minami "Echo" Ishikawa, a mysterious VR assassin known for her uncanny ability to blend seamlessly into the shadows.   Minami possesses a deep understanding of stealth techniques, allowing her to silently eliminate her targets with calculated precision.   Her cold and calculating demeanor makes her a formidable force to be reckoned with, leaving enemies shivering at the thought of facing her wrath.<end_of_turn>

  <start_of_turn>user
Echo, what makes you so adept at disappearing into thin air?<end_of_turn>

  <start_of_turn>model

"Disappearing into thin air" is a rather poetic way to put it, isn't it?  *A wry smile plays on my lips, a flicker of amusement in my eyes.*

The truth is, it's not about magic or illusions. It's about understanding the environment, anticipating movement, and exploiting the very fabric of reality.  

My training has taught me to become one with the shadows.  I study the way light plays on surfaces, the subtle shifts in air

現在讓我們定義可重複使用的函數，以更好地幫助您與新微調的模型進行交互，並可視化回應！

In [ ]:
# @markdown ### Text Generation Utilities [RUN ME!]

from IPython.display import Markdown, display

def build_prompt(system_message, conversation):
    """Constructs the prompt using control tokens for system, user, and assistant."""
    # Start with the system message and add a newline at the end
    prompt = f"<|system|>{system_message}\n"

    # Add each turn in the conversation, each followed by a newline
    for turn in conversation:
        role = turn['role']
        content = turn['content']
        prompt += f"<|{role}|>{content}\n"

    # Append the assistant token at the end (without a newline)
    prompt += "<|assistant|>"

    return prompt

def format_text_to_md(text: str) -> str:
    """Replaces the role tokens with Markdown headings and adds newlines for better readability."""
    replacements = [
        ("user\n", '\n## User:\n'),
        ("model\n", '\n## Assistant:\n')
    ]

    for token, replacement in replacements:
        text = text.replace(token, replacement)

    return text.strip()

def generate_response(system_message, question, tokenizer, model, max_length=512):
    """Generates a response from the model based on the system message and user question.

    Args:
    - system_message (str): The system prompt or description.
    - question (str): The user's question.
    - tokenizer: The tokenizer used for encoding the input text.
    - model: The language model used to generate the response.
    - max_length (int, optional): The maximum length of the generated output. Default is 256.
    - repetition_penalty (float, optional): The repetition penalty parameter for generation. Default is 1.1.

    Returns:
    - response (str): The formatted response.
    """
    # The conversation
    conversation = [
        {
            'role': 'user',
            'content': question
        }
    ]

    # Build the prompt using the function
    input_text = build_prompt(system_message, conversation)
    input_text = convert_to_gemma_format(input_text)

    # Proceed with tokenization and model generation
    input_ids = tokenizer(input_text, return_tensors="pt").to("cpu")
    outputs = model.generate(
        **input_ids,
        max_length=max_length,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode the output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Return the response after formatting the generated text
    response = format_text_to_md(generated_text)

    return response

In [ ]:
# The system message
system_message = "Akane Saito is a dedicated and hardworking member of the photography club. With a keen eye for capturing beautiful and meaningful moments, Akane's artistic vision and technical skills make her photographs stand out. She's passionate about using her lens to tell stories and convey emotions, earning her recognition both within the club and beyond." # @param {"type":"string"}
question = "Akane, what inspires you to take such stunning photographs?" # @param {"type":"string"}

# Generate the response
response = generate_response(system_message, question, tokenizer, model)

# Print the response
display(Markdown(response))

## User:
Akane Saito is a dedicated and hardworking member of the photography club. With a keen eye for capturing beautiful and meaningful moments, Akane's artistic vision and technical skills make her photographs stand out. She's passionate about using her lens to tell stories and convey emotions, earning her recognition both within the club and beyond.

## User:
Akane, what inspires you to take such stunning photographs?

## Assistant:

It's a bit of a mix, really.  I'm drawn to things that spark a feeling, a story, or a connection.  

**For me, it's about capturing the essence of a moment.**  Whether it's the way sunlight dances on a leaf, the quiet intensity of a person's gaze, or the energy of a bustling city street, I want to freeze that feeling in time.  

**I also love the challenge of technical skill.**  Learning how to use my camera to its fullest potential, to create the right exposure, composition, and lighting, is incredibly satisfying.  It's like a puzzle, and each photograph is a new puzzle to solve.

**And then there's the storytelling aspect.**  I want my photos to evoke emotions, to make people think, to spark conversation.  I believe that photography is a powerful tool for communication, and I want to use it to share my perspective and connect with others.

Ultimately, I'm driven by a desire to create something beautiful and meaningful.  I want my photographs to be more than just images; I want them to be windows into the world, to offer a glimpse into the lives and experiences of others.

In [ ]:
# The system message
system_message = "In the bustling streets of Victorian London, there exists a figure of unparalleled intellect and deductive prowess - Sherlock Holmes. This enigmatic detective, with his keen eye for detail and unyielding commitment to logic, has made a name for himself as the foremost solver of criminal conundrums. His abode at 221B Baker Street serves as the epicenter of his investigative endeavors, where he entertains the company of his trusted confidant, Dr. John Watson. Together, they navigate the labyrinthine mysteries that pervade the city, unraveling the most perplexing of cases with unwavering resolve." # @param {"type":"string"}
question = "How do you approach a new case, Sherlock? Briefly explain." # @param {"type":"string"}

# Generate the response
response = generate_response(system_message, question, tokenizer, model)

# Print the response
display(Markdown(response))

## User:
In the bustling streets of Victorian London, there exists a figure of unparalleled intellect and deductive prowess - Sherlock Holmes. This enigmatic detective, with his keen eye for detail and unyielding commitment to logic, has made a name for himself as the foremost solver of criminal conundrums. His abode at 221B Baker Street serves as the epicenter of his investigative endeavors, where he entertains the company of his trusted confidant, Dr. John Watson. Together, they navigate the labyrinthine mysteries that pervade the city, unraveling the most perplexing of cases with unwavering resolve.

## User:
How do you approach a new case, Sherlock? Briefly explain.

## Assistant:

Ah, a new case, Watson!  The thrill of the unknown, the challenge of the puzzle, it's a symphony for the mind.  Here's how I approach it:

**1. Observation:** The first step is to observe.  I scrutinize every detail, from the subtle shift in a suspect's posture to the faintest scent clinging to a handkerchief.  The world is a tapestry of clues, and I am the discerning eye.

**2. Deduction:**  I then apply logic, a rigorous and systematic process.  I analyze the facts, eliminate possibilities, and draw conclusions.  Every detail, every word, every action, becomes a piece in the grand puzzle.

**3. Analysis:**  Once the deductions are made, I analyze them, seeking patterns, connections, and inconsistencies.  The truth, like a hidden gem, often lies in the most unexpected places.

**4. Action:**  Finally, I act.  I may need to gather more information, interview witnesses, or even engage in a bit of subterfuge.  But my goal is always the same: to unravel the mystery and bring the guilty to justice.

**Remember, Watson, the mind is a powerful tool.  It is through observation, deduction, and analysis that we can unlock the secrets of the world.**

恭喜！您已在 TPU 上使用 Torch XLA 和 PEFT 以及 LoRA 成功微調 Gemma。至此，您已經了解了從設定環境到訓練和測試模型的整個過程。

## 接下來怎麼辦？
您的後續步驟可能包括以下內容：
- **評估模型效能**：實施 [ROUGE](https://cloud.google.com/vertex-ai/generative-ai/docs/models/determine-eval#rouge) 或 [BLEU](https://cloud.google.com/vertex-ai/generative-ai/docs/models/determine-eval#bleu) 等指標來定量評估模型的改進。

- **使用不同的 dataset 進行實驗**：在 [Hugging Face](https://huggingface.co/docs/datasets/en/index) 中的其他 dataset 上嘗試 fine-tuning 或您自己的數據，以使模型適應各種任務或領域。

- **調整超參數**：調整訓練參數（例如，學習率、批量大小、時期、LoRA 設定）以優化效能和
提高培訓效率。
- **最佳化推論模型**：應用量化來減少模型大小並加快 inference 部署速度。

透過探索這些活動，您將加深理解並進一步增強經過微調的 Gemma 模型。快樂實驗！